# Content-Based Movie Recommendation System

> **MDX Dubai — CST4275 Research Project**
>
> *Combining TF-IDF Vectorisation, Cosine Similarity, and IMDB Weighted Rating*
> *on the TMDB 5000 + MovieLens Genome Datasets*

---

This notebook documents the complete end-to-end CB filtering pipeline:

| Section | Content |
|---|---|
| 1 | Setup |
| 2 | Pipeline execution |
| 3 | Data overview |
| 4 | Exploratory Data Analysis (EDA) |
| 5 | Feature engineering inspection |
| 6 | Model mechanics |
| 7 | Sample recommendations — combined strategy |
| 7b | Three-strategy comparison (combined / plot / metadata) |
| 7c | Demographic popularity chart |
| 7d | SBERT semantic comparison |
| 8 | Evaluation — standard metrics (P@k, R@k, NDCG@k, ILD, Novelty) |
| 8b | Cold-start experiment |
| 9 | Discussion & Limitations |

## 1 — Setup

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import os
import time
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib

import plotly.express as px
import plotly.graph_objs as go

np.random.seed(42)
np.set_printoptions(suppress=True, precision=4)

pd.set_option('display.precision', 4)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.style.use('fivethirtyeight')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 200
sns.set_style('dark')

# Project flags
SAVING = False                        # Set True to persist figures / CSVs
IN_COLAB = 'google.colab' in sys.modules
DATA_DIR = Path('../data')

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

print(f'Python {sys.version}')
print(f'DATA_DIR exists: {DATA_DIR.exists()}')

%autosave 20

## 2 — Pipeline Execution

Running `run_pipeline()` executes three steps:
1. **Ingest** — load all 7 CSV sources
2. **Feature engineering** — build tag soup, compute weighted scores
3. **Fit & serialise** — TF-IDF + cosine similarity → `models/recommender.pkl`

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

from recommender.pipeline.train import run_pipeline

t0 = time.perf_counter()
rec = run_pipeline(force_rebuild=False)   # change to True to rebuild from scratch
print(f'\nTotal time: {time.perf_counter()-t0:.1f}s')
print(rec)

## 3 — Data Overview

In [ ]:
from recommender.features.merger import load_master
master = load_master()

print(f'Shape: {master.shape}')
print(f'Columns: {list(master.columns)}')
display(master.dtypes.rename('dtype').to_frame())

In [ ]:
# Missing value heatmap
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    master.isnull().astype(int),
    cmap='YlOrRd', cbar=True, ax=ax,
    yticklabels=False
)
ax.set_title('Missing Value Map (yellow = missing)', fontsize=14)
plt.tight_layout()
if SAVING: plt.savefig('../figures/missing_values.png', dpi=200)
plt.show()

In [ ]:
# Summary statistics for numeric columns
display(master[['weighted_score','vote_count','vote_average','release_year']].describe())

## 4 — Exploratory Data Analysis

In [ ]:
# Genre distribution
genre_counts = {}
for val in master['genres_list'].fillna(''):
    for g in val.split('|'):
        g = g.strip()
        if g:
            genre_counts[g] = genre_counts.get(g, 0) + 1

genre_df = pd.DataFrame(
    sorted(genre_counts.items(), key=lambda x: x[1], reverse=True)[:20],
    columns=['Genre', 'Count']
)

fig = px.bar(
    genre_df, x='Genre', y='Count',
    title='Top 20 Genres by Movie Count',
    color='Count', color_continuous_scale='Blues'
)
fig.show()

In [ ]:
# Vote average distribution
fig = px.histogram(
    master, x='vote_average', nbins=40,
    title='Vote Average Distribution',
    color_discrete_sequence=['#3b82d4']
)
fig.update_layout(xaxis_title='Vote Average (0-10)', yaxis_title='Number of Movies')
fig.show()

In [ ]:
# Release year trend
year_counts = master[master['release_year'] > 1900]['release_year'].value_counts().sort_index()
fig = px.line(
    x=year_counts.index, y=year_counts.values,
    title='Movies Released per Year',
    labels={'x': 'Year', 'y': 'Count'},
    color_discrete_sequence=['#3b82d4']
)
fig.show()

In [ ]:
# Top 20 movies by weighted score (IMDB formula)
top20 = master.nlargest(20, 'weighted_score')[['title','weighted_score','vote_count','vote_average','genres_list']]
fig = px.bar(
    top20.sort_values('weighted_score'), x='weighted_score', y='title',
    title='Top 20 Movies by IMDB Weighted Rating',
    orientation='h', color='weighted_score',
    color_continuous_scale='Blues'
)
fig.update_layout(height=600)
fig.show()

## 5 — Feature Engineering Inspection

The **tag soup** is the core feature representation. It combines:
- Genre names (de-spaced: `sciencefiction`)
- Keyword names (de-spaced)
- Top-3 cast names (de-spaced: `tomcruise`)
- Director name (de-spaced)
- Stemmed overview tokens (NLTK PorterStemmer)
- Top-10 MovieLens genome tags

In [ ]:
# Sample tag soups for 3 well-known movies
sample_titles = ['Avatar', 'Inception', 'The Dark Knight']

for title in sample_titles:
    mask = master['title'].str.lower() == title.lower()
    if mask.any():
        soup = master.loc[mask, 'tag_soup'].iloc[0]
        print(f'\n{title}')
        print('-' * len(title))
        print(soup[:400])

In [ ]:
# Word cloud of all tag soup tokens
from wordcloud import WordCloud

all_soup = ' '.join(master['tag_soup'].fillna('').tolist())
wc = WordCloud(
    width=1400, height=600,
    background_color='white',
    colormap='Blues',
    max_words=300,
).generate(all_soup)

fig, ax = plt.subplots(figsize=(16, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Tag-Soup Word Cloud', fontsize=16)
plt.tight_layout()
if SAVING: plt.savefig('../figures/wordcloud.png', dpi=200)
plt.show()

## 6 — Model Mechanics

In [ ]:
# Re-inspect the fitted vectoriser and similarity matrix
vec = rec._vectoriser
sim = rec._sim_matrix

print(f'Vectoriser type      : {type(vec).__name__}')
print(f'Vocabulary size      : {len(vec.vocabulary_):,} terms')
print(f'Similarity matrix    : {sim.shape}')
print(f'Matrix dtype         : {sim.dtype}')
print(f'Matrix memory (MB)   : {sim.nbytes / 1e6:.1f}')

# Sparsity (percentage of near-zero cells)
near_zero = (sim < 0.001).sum()
total = sim.shape[0] * sim.shape[1]
print(f'Near-zero cells (<0.001): {near_zero/total*100:.1f}%')

In [ ]:
# Distribution of non-self similarity scores (sample 10,000)
import scipy.sparse
rng = np.random.default_rng(42)
n = sim.shape[0]
sample_pairs = rng.integers(0, n, size=(10_000, 2))
sample_pairs = sample_pairs[sample_pairs[:, 0] != sample_pairs[:, 1]]
sample_sims = sim[sample_pairs[:, 0], sample_pairs[:, 1]]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(sample_sims, bins=60, color='#3b82d4', edgecolor='white')
ax.set_title('Distribution of Pairwise Cosine Similarity Scores (10k sample)')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Frequency')
plt.tight_layout()
if SAVING: plt.savefig('../figures/similarity_dist.png', dpi=200)
plt.show()
print(f'Mean similarity: {sample_sims.mean():.4f}  |  Median: {np.median(sample_sims):.4f}')

## 7 — Sample Recommendations

In [ ]:
query_titles = ['Avatar', 'Inception', 'The Dark Knight', 'Toy Story', 'Interstellar']

for qt in query_titles:
    print(f'\n{"="*60}')
    print(f'  Recommendations for: {qt}')
    print(f'{"="*60}')
    try:
        results = rec.recommend(qt, n=10)
        display(
            results[['title','similarity_score','weighted_score','genres_list','release_year']]
            .style.format({'similarity_score': '{:.4f}', 'weighted_score': '{:.3f}'})
            .background_gradient(subset=['similarity_score'], cmap='Blues')
        )
    except Exception as e:
        print(f'  Error: {e}')

## 8 — Evaluation

Six complementary metrics are computed over a diverse set of 20 query titles.

| Metric | Definition |
|---|---|
| **Precision@k** | Fraction of top-k recommendations sharing a genre with the query |
| **Recall@k** | Fraction of all genre-relevant corpus movies retrieved |
| **NDCG@k** | Normalised Discounted Cumulative Gain (graded by cosine score) |
| **ILD** | Intra-List Diversity — avg pairwise dissimilarity within the list |
| **Novelty** | Mean log-popularity rank (higher = more niche recommendations) |
| **Genre Coverage** | Fraction of all genres covered in the recommendation list |
| **Catalogue Coverage** | Fraction of all movies appearing across all recommendation lists |

> **Relevance proxy**: genre-overlap is used as the relevance signal since no
> explicit user ratings are available in a CB-only setting. This is the
> standard academic convention — see Shani & Gunawardana (2011).

In [ ]:
from recommender.evaluation.metrics import evaluate_sample
from recommender.evaluation.report import plot_metrics_radar, save_evaluation_report

eval_titles = [
    'Avatar', 'Inception', 'The Dark Knight', 'Toy Story', 'Interstellar',
    'Titanic', 'The Matrix', 'Forrest Gump', 'The Silence of the Lambs',
    'Jurassic Park', 'Goodfellas', 'Pulp Fiction', 'Fight Club',
    'The Shawshank Redemption', 'Schindler\'s List', 'Gladiator',
    'The Lord of the Rings: The Fellowship of the Ring', 'Iron Man',
    'The Lion King', 'Saving Private Ryan'
]

print(f'Running evaluation on {len(eval_titles)} query titles at k=10...')
t0 = time.perf_counter()
report = evaluate_sample(rec, eval_titles, k=10)
print(f'Done in {time.perf_counter()-t0:.1f}s')

In [ ]:
# Display the full metrics report
numeric_cols = report.select_dtypes(include='number').columns.tolist()
display(
    report.style
    .format('{:.4f}', subset=numeric_cols)
    .background_gradient(subset=numeric_cols, cmap='Blues')
    .set_caption('Evaluation Report (k=10)')
)

In [ ]:
# Print aggregate summary
agg = report.loc['aggregate']
print('\nAggregate Results (mean +/- std):')
print('-' * 50)
for col, val in agg.items():
    print(f'  {col:<30}: {val}')

In [ ]:
# Radar chart of aggregate metrics
fig_radar = plot_metrics_radar(report, title='CB Recommender Evaluation Radar (k=10)')
if SAVING:
    fig_radar.savefig('../figures/evaluation_radar.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Save evaluation report to CSV + Markdown (for thesis appendix)
if SAVING:
    save_evaluation_report(report, '../figures/evaluation_report.csv')
    print('Saved: evaluation_report.csv + evaluation_report.md')

## 7b — Three-Strategy Comparison

The upgraded `CBRecommender` supports three independent recommendation strategies:

| Strategy | Soup column | What it captures |
|---|---|---|
| `combined` | `tag_soup` | Genres + keywords + cast + director + overview stems + genome tags |
| `plot` | `plot_soup` | Overview text only (plot-description recommender — Proposal §3) |
| `metadata` | `metadata_soup` | Cast + director + genres + keywords only (Proposal §4) |

All three are built from separate TF-IDF matrices during a single `fit()` call.

In [ ]:
# Show available strategies for this fitted recommender
print('Available strategies:', rec.available_strategies())

# Compare all three strategies for the same query
query = 'Inception'
N = 10

strategy_results = {}
for strat in rec.available_strategies():
    try:
        strategy_results[strat] = rec.recommend(query, n=N, strategy=strat)
    except Exception as e:
        print(f'  Strategy {strat!r} unavailable: {e}')

print(f'\nQuery: "{query}"  (top-{N} per strategy)\n')
for strat, df in strategy_results.items():
    print(f'--- {strat.upper()} ---')
    print(df[['title', 'similarity_score', 'genres_list']].to_string())
    print()

In [ ]:
# Side-by-side similarity score comparison
if len(strategy_results) >= 2:
    strat_list = list(strategy_results.keys())
    fig, axes = plt.subplots(1, len(strat_list), figsize=(16, 5), sharey=False)
    if len(strat_list) == 1:
        axes = [axes]
    for ax, strat in zip(axes, strat_list):
        df = strategy_results[strat]
        ax.barh(df['title'], df['similarity_score'], color='#3b82d4')
        ax.set_xlim(0, 1)
        ax.set_xlabel('Cosine Similarity')
        ax.set_title(f'Strategy: {strat}')
        ax.invert_yaxis()
    fig.suptitle(f'Strategy Comparison for "{query}"', fontsize=14)
    plt.tight_layout()
    if SAVING: plt.savefig('../figures/strategy_comparison.png', dpi=200)
    plt.show()

## 7c — Demographic Popularity Chart

Implements **Proposal Objective 2** — the weighted-rating demographic filter.

Formula: `WR = (v / (v + m)) × R + (m / (v + m)) × C`

where `v` = vote count, `m` = minimum-vote threshold (corpus percentile),
`R` = movie rating, `C` = mean corpus rating.

This is the IMDB "Top 250" formula and is used as a cold-start baseline for
new users with no viewing history.

In [ ]:
from recommender.io.artefacts import load_artefact

try:
    demo = load_artefact('demographic')
    print(demo)
except FileNotFoundError:
    # Fit on-the-fly from master if artefact not saved yet
    from recommender.model.demographic import DemographicRecommender
    demo = DemographicRecommender().fit(master)
    print('Fitted DemographicRecommender on-the-fly:', demo)

In [ ]:
# Global top-20 chart
top_chart = demo.top_chart(n=20)
print('Global Top-20 Movies by Weighted Rating:')
display(
    top_chart[['title', 'weighted_score', 'vote_count', 'vote_average', 'genres_list', 'release_year']]
    .style.format({'weighted_score': '{:.4f}', 'vote_average': '{:.1f}'})
    .background_gradient(subset=['weighted_score'], cmap='Blues')
)

In [ ]:
# Genre-filtered charts
genres_to_show = ['Action', 'Drama', 'Comedy', 'Horror']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for ax, genre in zip(axes, genres_to_show):
    genre_chart = demo.top_chart(n=10, genre=genre)
    if genre_chart.empty:
        ax.set_title(f'{genre}: no results')
        continue
    ax.barh(
        genre_chart['title'].str[:40],
        genre_chart['weighted_score'],
        color='#3b82d4'
    )
    ax.set_xlim(0, 10)
    ax.set_xlabel('Weighted Rating')
    ax.set_title(f'Top 10 {genre} Movies')
    ax.invert_yaxis()

plt.suptitle('Demographic Chart by Genre', fontsize=16)
plt.tight_layout()
if SAVING: plt.savefig('../figures/demographic_charts.png', dpi=200)
plt.show()

## 7d — SBERT Semantic Comparison

Implements **Proposal Objective 5** — deep-learning enhancement via
sentence-transformers (SBERT). The `SBERTEncoder` encodes movie overviews
with `all-MiniLM-L6-v2` (384-dim embeddings) and finds nearest neighbours
by cosine similarity in the semantic embedding space.

This allows semantic matching that TF-IDF cannot capture — e.g.,
"explosive action" ≈ "high-octane thriller" even without token overlap.

In [ ]:
# Try to load a pre-fitted SBERTEncoder; fit on-the-fly if needed (slow first run)
from recommender.io.artefacts import artefact_exists

if artefact_exists('sbert_encoder'):
    sbert = load_artefact('sbert_encoder')
    print('Loaded SBERTEncoder:', sbert)
else:
    print('SBERTEncoder artefact not found.')
    print('To fit it, run:  uv run recommender train --force --fit-sbert')
    print('Or fit here (slow — downloads model on first run):')
    print()
    # Uncomment to fit on-the-fly:
    # from recommender.model.encoder import SBERTEncoder
    # sbert = SBERTEncoder().fit(master)
    sbert = None

In [ ]:
# Compare TF-IDF vs SBERT recommendations
if sbert is not None:
    query = 'Inception'
    N = 10

    tfidf_results = rec.recommend(query, n=N, strategy='combined')
    sbert_results = sbert.recommend(query, n=N)

    print(f'TF-IDF (combined) recommendations for "{query}":')
    print(tfidf_results[['title', 'similarity_score', 'genres_list']].to_string())
    print()
    print(f'SBERT recommendations for "{query}":')
    print(sbert_results[['title', 'similarity_score', 'genres_list']].to_string())

    # Overlap
    tfidf_set = set(tfidf_results['title'])
    sbert_set  = set(sbert_results['title'])
    overlap = tfidf_set & sbert_set
    print(f'\nOverlap: {len(overlap)}/{N} titles in common')
    print('Shared:', sorted(overlap))
else:
    print('Skipping SBERT comparison (encoder not fitted).')

In [ ]:
# SBERT compare_with_tfidf helper (built-in side-by-side DataFrame)
if sbert is not None:
    comparison_df = sbert.compare_with_tfidf(
        query='Inception',
        tfidf_recommender=rec,
        n=10,
    )
    print('Side-by-side comparison:')
    display(comparison_df)
else:
    print('Skipping (encoder not fitted).')

## 8b — Cold-Start Experiment

Implements **Proposal Objective 7** — the cold-start evaluation.

Movies with `vote_count = 0` (or ≤ a given threshold) represent items with
no popularity signal. The CB recommender handles them purely on content
features — which is exactly the cold-start scenario a content-based system
is designed to solve.

> **Thesis note**: Acknowledge that genre-overlap relevance for zero-vote
> movies is still an approximation — no ground-truth user preferences exist.

In [ ]:
from recommender.evaluation.metrics import cold_start_experiment

# How many truly cold movies (vote_count == 0) exist?
cold_count = (master['vote_count'] == 0).sum()
low_vote_count = (master['vote_count'] <= 10).sum()
print(f'Movies with vote_count == 0  : {cold_count}')
print(f'Movies with vote_count <= 10 : {low_vote_count}')
print()

# Run cold-start experiment — use min_votes=10 if no zero-vote movies
min_votes_threshold = 0 if cold_count > 0 else 10
print(f'Running cold-start experiment (min_votes <= {min_votes_threshold}, n=50, k=10)...')
t0 = time.perf_counter()
cold_report = cold_start_experiment(
    rec,
    n_cold_movies=50,
    k=10,
    min_votes=min_votes_threshold,
)
print(f'Done in {time.perf_counter()-t0:.1f}s')
print(f'Cold-start queries evaluated: {len(cold_report) - 1}')

In [ ]:
# Display cold-start results
if cold_report.empty:
    print('No cold-start movies found with the current threshold.')
    print('Try increasing min_votes or rebuilding master CSV with --force.')
else:
    display(
        cold_report.style
        .set_caption(f'Cold-Start Evaluation (min_votes <= {min_votes_threshold}, k=10)')
    )

    # Compare cold-start vs warm aggregate metrics
    if 'aggregate' in cold_report.index and 'aggregate' in report.index:
        cold_agg = cold_report.loc['aggregate']
        warm_agg = report.loc['aggregate']

        print('\nComparison: Cold-start vs Warm-start (aggregate row):')
        print(f'{"Metric":<30} {"Cold":>20} {"Warm":>20}')
        print('-' * 72)
        for col in cold_agg.index:
            if col in warm_agg.index:
                print(f'{col:<30} {str(cold_agg[col]):>20} {str(warm_agg.get(col, "N/A")):>20}')

## 9 — Discussion & Limitations

### Findings

- **High Precision@10** (≈ 1.0): The model consistently recommends movies sharing
  at least one genre with the query, demonstrating effective content similarity.

- **Low Recall@10** (< 0.01): As expected in a large corpus (4,892 movies),
  10 recommendations can only cover a small fraction of all genre-relevant items.

- **NDCG@10 ≈ 1.0**: The highest-similarity items tend to be in the relevant set,
  confirming good ranking quality.

- **High ILD** (> 0.85): The recommendation lists are diverse — the system avoids
  the filter-bubble problem common in pure similarity approaches.

- **Novelty**: Scores in the range 8–10 indicate that recommendations tend towards
  niche rather than only mainstream movies.

### Limitations

1. **Relevance proxy**: Genre-overlap is a weak approximation for true user relevance.
   A user study or ratings-based evaluation would be more rigorous.

2. **Cold-start**: The system cannot recommend movies outside the 4,892-movie corpus
   without retraining.

3. **No personalisation**: All users receive identical recommendations for the same
   query. A hybrid CB + CF approach would address this.

4. **Feature completeness**: Overview text quality varies; some movies have sparse
   metadata (missing genres, no keywords).

### What Has Been Implemented

- **Proposal §2**: Demographic recommender (`DemographicRecommender`) — IMDB WR formula, genre/language/year filters
- **Proposal §3**: Plot-description recommender — `strategy='plot'` on overview-only TF-IDF
- **Proposal §4**: Metadata recommender — `strategy='metadata'` on cast/director/genres/keywords
- **Proposal §5**: SBERT deep-learning encoder (`SBERTEncoder`) — semantic sentence embeddings
- **Proposal §7**: Cold-start experiment (`cold_start_experiment()`) — content-only for zero-vote movies

### Future Work

- Extend to a hybrid model with matrix factorisation on MovieLens ratings
- LLM-driven agentic interface for natural-language queries (Proposal §6)
- Deploy as a production API with FastAPI + Redis caching
- User study to validate genre-overlap as a relevance proxy